# HRS Silver CDM DML Functional Specification

________________________________________
## Document Information
|Document Name:|HRS Silver CDM DDL Functional Specification|
| --- | --- |
|Version:|1.0|
|Author:|Perez|
|Last Updated:|2026-07-30|
|Target Runtime:|Databricks Runtime 15.x|
|SQL Dialect:|Spark SQL|
|Storage Format:|Delta Lake|
|Deployment Environment:|Development|
________________________________________

## 1. Objective
Define the rules, transformations, joins, and loading logic required to generate a Databricks SQL DML script that loads data into one Silver CDM table for a single RAND HRS survey section.

The generated SQL must be deterministic, reproducible, and fully aligned with the DDL specification.
________________________________________

## 2. Purpose
This specification describes:

- Source dataset requirements
- Join logic for foreign keys
- Business key derivation
- Column‑level transformation rules
- Load pattern (Insert‑Only)
- SQL generation standards

The output of this specification is a DML SQL statement that inserts transformed records into the Silver CDM table.
________________________________________

## 3. Source Dataset

| Parameter | Value |
| --- | --- |
| SOURCE_CATALOG | dev_catalog |
| SOURCE_SCHEMA | brz_raw_hrs |
| SOURCE_TABLE | randhrs1992_2022v1 |
| FULL_SOURCE_NAME | dev_catalog.brz_raw_hrs.randhrs1992_2022v1 |
________________________________________

## 4. Target Table

| Parameter | Value |
| --- | --- |
| TARGET_CATALOG | dev_catalog |
| TARGET_SCHEMA | slv_cdm_hrs |
| TARGET_TABLE | hrs_demographics |
| FULL_TARGET_NAME | dev_catalog.slv_cdm_hrs.hrs_demographics |
| LOAD_PATTERN | Insert Only |
________________________________________

## 5. Required Joins
### 5.1 Respondent Foreign Key Join
To populate respondent_id, join the target table to the parent table:

JOIN dev_catalog.slv_cdm_hrs.hrs_respondent r
  ON r.hhidpn = src.hhidpn

### 5.2 Wave Foreign Key Join

To populate wave_id, join the target table to the parent table:

JOIN dev_catalog.slv_cdm_hrs.hrs_wave w
  ON w.wave_number = <derived_wave_number>
________________________________________

## 6. Identifier Column Rules

### 6.1 hhidpn

- Populated directly from the source dataset
- Source column: HHIDPN
- Nullable: Yes

### 6.2 wave_number

Derived from the Source Variable column in the Source‑to‑Target Mapping Matrix.

Rule:  
Extract the two‑digit wave number from the Source Variable name.

Examples:

| Source Variable | Derived wave_number |
| --- | --- |
| R1AGEY_E | 1 |
| R02AGEY_E | 2 |
| R16CENREG | 16 |

Extraction Logic:
- Strip leading “R”
- Read the next 1–2 digits
- Cast to SMALLINT
________________________________________

## 7. Foreign Key Population Rules

### 7.1 respondent_id

- Derived by joining hrs_respondent on hhidpn
- Required (NOT NULL)
- If no match exists → record must be excluded

### 7.2 wave_id

- Derived by joining hrs_wave on wave_number
- Required (NOT NULL)
- If no match exists → record must be excluded
________________________________________

## 8. Business Key Rules

| Column | Rule |
| --- | --- |
| respondent_id | Required |
| wave_id | Required |

Together they uniquely identify one demographic observation.

Duplicate combinations must not be inserted.
________________________________________

### 9. Transformation Rules

### 9.1 RAND Variable Type Transformations

| RAND Type | Rule |
| --- | --- |
| CONT | TRY_CAST(source AS DECIMAL(10,2)) |
| CATEG | TRY_CAST(source AS INT) |
| CHAR | TRY_CAST(source AS STRING) |

### 9.2 Column‑Level Transformations

Each target column uses the transformation defined in the Source‑to‑Target Mapping Matrix.

Examples:

| Target Column | Source Variable | Transformation |
| --- | --- | --- |
| agey_e | R1AGEY_E | TRY_CAST(R1AGEY_E AS DECIMAL(10,2)) |
| raracem | RARACEM | TRY_CAST(RARACEM AS INT) |
| cenreg | R1CENREG | TRY_CAST(R1CENREG AS INT) |
________________________________________

## 10. Load Pattern

Insert‑Only

- No updates
- No deletes
- No merges
- No upserts
- All loads append new records.
________________________________________

## 11. DML SQL Generation Requirements

| Requirement | Required |
| --- | --- |
| INSERT INTO … SELECT | Yes |
| Joins for FK resolution | Yes |
| Wave number derivation | Yes |
| Column‑level transformations | Yes |
| Business key enforcement | Yes |
| Exclude records missing FK matches | Yes |
| Uppercase SQL keywords | Yes |
| Consistent indentation | Yes |
| No unsupported operations | Yes |

Unsupported operations:

- UPDATE
- DELETE
- MERGE
- UPSERT
- ANALYZE
- OPTIMIZE
- ZORDER

________________________________________

## 12. Wave Expansion and Unpivot Rules (Required for Wide RAND HRS Source)

### 12.1 Problem Description
The RAND HRS raw dataset (dev_catalog.brz_raw_hrs.randhrs1992_2022v1) stores all waves in a single row per HHIDPN, using wave‑specific column names such as:

- R1AGEY_E, R2AGEY_E, … R16AGEY_E
- R1CENREG, R2CENREG, … R16CENREG
- R1MSTAT, R2MSTAT, … R16MSTAT

However, the Silver CDM target table requires:

One row per respondent per wave.

Therefore, the DML must expand each source row into up to 16 target rows, one for each wave where data exists.

### 12.2 Wave Expansion Rule
For every target column defined in the Source‑to‑Target Mapping Matrix:

1. Identify all wave‑specific source variables (e.g., R1AGEY_E … R16AGEY_E).
2. Generate 16 SELECT branches, one per wave.
3. UNION ALL the branches to produce a long dataset.

This produces a structure like:

SELECT HHIDPN, 1 AS wave_number, R1AGEY_E AS agey_e, R1CENREG AS cenreg, R1MSTAT AS mstat, ...
UNION ALL
SELECT HHIDPN, 2 AS wave_number, R2AGEY_E AS agey_e, R2CENREG AS cenreg, R2MSTAT AS mstat, ...
...
UNION ALL
SELECT HHIDPN, 16 AS wave_number, R16AGEY_E AS agey_e, R16CENREG AS cenreg, R16MSTAT AS mstat, ...

### 12.3 Multi‑Wave Variable Rule
A variable is considered multi‑wave if the Source‑to‑Target Mapping Matrix lists more than one wave.

Examples:
| Target Column | Multi‑Wave? | Waves |
| --- | --- | --- |
| agey_e | Yes | 1–16 |
| cenreg | Yes | 1–16 |
| mstat | Yes | 1–16 |
| raracem | No | 1 |
| rahispan | No | 1 |
| rarelig | No | 1 |
| ravetrn | No | 1 |

Rule:  
Multi‑wave variables must be included in each wave’s SELECT branch using the wave‑specific source variable.

Single‑wave variables must be included in all wave branches using the same source variable.

### 2.4 Wave Number Derivation Rule
Wave number is not derived from the source variable name anymore.

Instead:

- Wave number is explicitly defined by the SELECT branch index (1–16).

This ensures consistency and avoids parsing errors.

### 12.5 Foreign Key Resolution Rule (Updated)
respondent_id
Join using HHIDPN:

JOIN hrs_respondent r
  ON r.hhidpn = src.HHIDPN

wave_id
Join using the expanded wave_number:

JOIN hrs_wave w
  ON w.wave_number = expanded.wave_number

### 12.6 Business Key Enforcement

The DML must ensure:

UNIQUE (respondent_id, wave_id)
This is enforced by:

- Excluding NULL FK matches
- Deduplicating after expansion

### 12.7 Required DML Output Structure (Updated)

The generated DML must follow this structure:

1. UNION ALL block that expands 1 row → 16 rows
2. Transformations applied inside each SELECT branch
3. FK joins applied after expansion
4. INSERT INTO target table
5. Deduplication using ROW_NUMBER()
________________________________________

## 13. DML Output Structure

The generated SQL must follow this structure:

1. INSERT INTO <FULL_TARGET_NAME>
2. Column list in target table order
3. SELECT block containing:
- Identifier columns
- Audit columns
- Business columns
- FK columns
4. Joins to parent tables
5. Wave number derivation logic
6. Transformations per mapping matrix
7. WHERE clause enforcing FK existence
8. Optional deduplication 
________________________________________

## 14. Validation Requirements

| Validation | Required |
| --- | --- |
| respondent_id resolved | ✓ |
| wave_id resolved | ✓ |
| No duplicate respondent_id + wave_id | ✓ |
| All NOT NULL columns populated | ✓ |
| All transformations applied | ✓ |
| All joins valid | ✓ |
________________________________________

## 15. Deliverables

| Item | Value |
| --- | --- |
| DML File | ``/sql/dml/load_<TABLE_NAME>.sql`` |
| Output | SQL Only |

________________________________________

